In [39]:
import pandas as pd

In [40]:
df_bronze_meta = pd.read_csv("../data/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv")

In [41]:
print(df_bronze_meta.shape)
print(df_bronze_meta.columns.tolist())

(10704, 13)
['ano', 'id_municipio', 'rede', 'taxa_alfabetizacao', 'meta_alfabetizacao_2024', 'meta_alfabetizacao_2025', 'meta_alfabetizacao_2026', 'meta_alfabetizacao_2027', 'meta_alfabetizacao_2028', 'meta_alfabetizacao_2029', 'meta_alfabetizacao_2030', 'nivel_alfabetizacao', 'percentual_participacao']


In [42]:
# Cobertura por ano
print(
    df_bronze_meta.groupby("ano")["id_municipio"]
    .nunique()
)

ano
2023    5352
2024    5352
Name: id_municipio, dtype: int64


In [43]:
# Dado que SP foi o estado que chamou mais atenção pela ausência de dados, vamos analisar ele mais a fundo com esse enriquecimento
sp_2023 = df_bronze_meta[
    (df_bronze_meta["ano"] == 2023) &
    (df_bronze_meta["id_municipio"].astype(str).str.startswith("35"))
]

print("Municípios SP na camada bronze usando dados consolidados em 2023:", sp_2023["id_municipio"].nunique())

Municípios SP na camada bronze usando dados consolidados em 2023: 626


In [44]:
# Dado que SP foi o estado que chamou mais atenção pela ausência de dados, vamos analisar ele mais a fundo com esse enriquecimento
sp_2024 = df_bronze_meta[
    (df_bronze_meta["ano"] == 2023) &
    (df_bronze_meta["id_municipio"].astype(str).str.startswith("35"))
]

print("Municípios SP na camada bronze usando dados consolidados em 2024:", sp_2024["id_municipio"].nunique())

Municípios SP na camada bronze usando dados consolidados em 2024: 626


In [45]:
df_2024 = df_bronze_meta[df_bronze_meta["ano"] == 2024].copy()

# Municípios com taxa de alfabetização
com_taxa = df_2024[
    df_2024["taxa_alfabetizacao"].notna()
]["id_municipio"].nunique()

# Municípios com meta 2024
com_meta = df_2024[
    df_2024["meta_alfabetizacao_2024"].notna()
]["id_municipio"].nunique()

# Municípios com as duas informações
com_taxa_e_meta = df_2024[
    df_2024["taxa_alfabetizacao"].notna() &
    df_2024["meta_alfabetizacao_2024"].notna()
]["id_municipio"].nunique()

print("Com taxa 2024:", com_taxa)
print("Com meta 2024:", com_meta)
print("Com taxa + meta:", com_taxa_e_meta)

Com taxa 2024: 5352
Com meta 2024: 5232
Com taxa + meta: 5232


### Comparar agora SP com os dados da base analítica

In [46]:
import sys
sys.path.append("..")

from src.analytical.build_base_analitica import build_base_analitica



In [47]:
import pandas as pd

dim_municipio = pd.read_parquet(
    "../data/gold/dimensions/dim_municipio/dim_municipio.parquet"
)

df = pd.read_parquet(
    "../data/gold/facts/fato_alfabetizacao_municipio/fato_alfabetizacao_municipio.parquet"
)

In [48]:
df_analitica = build_base_analitica(
    df,
    dim_municipio
)

In [49]:
ids_base = set(df_analitica["id_municipio"].astype(str))
ids_novo = set(df_bronze_meta["id_municipio"].astype(str))

print("Municípios em comum:", len(ids_base & ids_novo))
print("Somente na base nova:", len(ids_novo - ids_base))
print("Somente na base analítica:", len(ids_base - ids_novo))

Municípios em comum: 4611
Somente na base nova: 741
Somente na base analítica: 0


### Comparando a nova base com a GOLD

In [50]:
df_bronze_meta[
    df_bronze_meta["ano"] == 2023
]["rede"].value_counts(dropna=False)

rede
Municipal    5352
Name: count, dtype: int64

In [51]:
df_bronze_meta[
    df_bronze_meta["ano"] == 2024
]["rede"].value_counts(dropna=False)

rede
Municipal    5352
Name: count, dtype: int64

In [52]:
ids_base = set(
    df_analitica["id_municipio"].astype(str)
)

df_bronze_meta["id_municipio"] = (
    df_bronze_meta["id_municipio"].astype(str)
)

df_adicionais = df_bronze_meta[
    ~df_bronze_meta["id_municipio"].isin(ids_base)
].copy()

In [53]:
df_adicionais.groupby("ano").agg(
    municipios=("id_municipio", "nunique"),
    com_taxa=("taxa_alfabetizacao", lambda x: x.notna().sum()),
    com_nivel=("nivel_alfabetizacao", lambda x: x.notna().sum()),
    com_participacao=("percentual_participacao", lambda x: x.notna().sum())
)

,municipios,com_taxa,com_nivel,com_participacao
ano,,,,
2023,741,621,621,621
2024,741,741,741,741


In [54]:
for ano in [2023, 2024]:

    temp = df_adicionais[df_adicionais["ano"] == ano]

    print(f"\n===== {ano} =====")

    print(
        "Municípios:",
        temp["id_municipio"].nunique()
    )

    print(
        "Com taxa:",
        temp.loc[
            temp["taxa_alfabetizacao"].notna(),
            "id_municipio"
        ].nunique()
    )

    print(
        "Com nível:",
        temp.loc[
            temp["nivel_alfabetizacao"].notna(),
            "id_municipio"
        ].nunique()
    )

    print(
        "Com participação:",
        temp.loc[
            temp["percentual_participacao"].notna(),
            "id_municipio"
        ].nunique()
    )


===== 2023 =====
Municípios: 741
Com taxa: 621
Com nível: 621
Com participação: 621

===== 2024 =====
Municípios: 741
Com taxa: 741
Com nível: 741
Com participação: 741


In [55]:
temp_2024 = df_adicionais[
    df_adicionais["ano"] == 2024
]

print(
    "Adicionais com taxa + meta:",
    temp_2024.loc[
        temp_2024["taxa_alfabetizacao"].notna() &
        temp_2024["meta_alfabetizacao_2024"].notna(),
        "id_municipio"
    ].nunique()
)

Adicionais com taxa + meta: 621


In [56]:
ids_621 = set(
    temp_2024.loc[
        temp_2024["taxa_alfabetizacao"].notna() &
        temp_2024["meta_alfabetizacao_2024"].notna(),
        "id_municipio"
    ]
)

temp_2023 = df_adicionais[
    (df_adicionais["ano"] == 2023) &
    (df_adicionais["id_municipio"].isin(ids_621))
]

print(
    "621 municípios com dados em 2023:",
    temp_2023["id_municipio"].nunique()
)

print(
    "621 municípios sem dados em 2023:",
    len(ids_621 - set(temp_2023["id_municipio"]))
)

621 municípios com dados em 2023: 621
621 municípios sem dados em 2023: 0


In [57]:
temp_2023[[
    "taxa_alfabetizacao",
    "nivel_alfabetizacao",
    "percentual_participacao"
]].notna().sum()

taxa_alfabetizacao         621
nivel_alfabetizacao        621
percentual_participacao    621
dtype: int64

In [58]:
temp_2023[[
    "taxa_alfabetizacao",
    "nivel_alfabetizacao",
    "percentual_participacao"
]].notna().sum()

taxa_alfabetizacao         621
nivel_alfabetizacao        621
percentual_participacao    621
dtype: int64

In [59]:
temp_2023[[
    "taxa_alfabetizacao",
    "nivel_alfabetizacao",
    "percentual_participacao"
]].describe()

,taxa_alfabetizacao,nivel_alfabetizacao,percentual_participacao
count,621.000000,621.000000,621.000000
mean,60.890338,2.542673,91.360274
std,14.616812,1.364362,4.565748
min,23.900000,0.000000,74.070000
25%,50.200000,2.000000,88.460000
50%,59.200000,2.000000,91.860000
75%,69.500000,3.000000,94.550000
max,100.000000,5.000000,100.000000


### Comparando com os dados da GOLD

In [60]:
# Dados de 2023 do CSV para os municípios da V1
csv_2023 = df_bronze_meta[
    (df_bronze_meta["ano"] == 2023) &
    (df_bronze_meta["id_municipio"].isin(
        df_analitica["id_municipio"].astype(str)
    ))
][[
    "id_municipio",
    "taxa_alfabetizacao"
]].copy()

csv_2023 = csv_2023.rename(
    columns={
        "taxa_alfabetizacao": "taxa_alfabetizacao_csv_2023"
    }
)

comparacao = df_analitica.merge(
    csv_2023,
    on="id_municipio",
    how="inner"
)

comparacao["diferenca_taxa"] = (
    comparacao["taxa_alfabetizacao_2023"]
    - comparacao["taxa_alfabetizacao_csv_2023"]
)

comparacao["diferenca_taxa"].describe()

count    4611.000000
mean       -0.005619
std         0.153474
min        -3.960000
25%        -0.030000
50%         0.000000
75%         0.020000
max         1.970000
Name: diferenca_taxa, dtype: float64

In [61]:
print(
    "Municípios com diferença:",
    (comparacao["diferenca_taxa"].abs() > 0.0001).sum()
)

print(
    "Maior diferença absoluta:",
    comparacao["diferenca_taxa"].abs().max()
)

Municípios com diferença: 4089
Maior diferença absoluta: 3.960000000000001


In [62]:
comparacao[
    [
        "id_municipio",
        "taxa_alfabetizacao_2023",
        "taxa_alfabetizacao_csv_2023",
        "diferenca_taxa"
    ]
].sort_values(
    "diferenca_taxa",
    key=lambda x: x.abs(),
    ascending=False
).head(20)

,id_municipio,taxa_alfabetizacao_2023,taxa_alfabetizacao_csv_2023,diferenca_taxa
3867,4308300,58.14,62.1,-3.96
725,2205854,58.69,62.1,-3.41
1705,2804904,13.83,16.9,-3.07
52,1300029,36.87,39.9,-3.03
1686,2803104,17.19,19.4,-2.21
1682,2802700,24.94,27.0,-2.06
4330,5107107,92.07,90.1,1.97
4185,5004007,80.80,78.9,1.90
72,1302108,75.46,77.3,-1.84
4228,5008404,67.11,65.4,1.71


In [63]:
comparacao[
    [
        "id_municipio",
        "taxa_alfabetizacao_2023",
        "taxa_alfabetizacao_csv_2023",
        "diferenca_taxa"
    ]
].sort_values(
    "diferenca_taxa",
    key=lambda x: x.abs(),
    ascending=False
).head(20)

,id_municipio,taxa_alfabetizacao_2023,taxa_alfabetizacao_csv_2023,diferenca_taxa
3867,4308300,58.14,62.1,-3.96
725,2205854,58.69,62.1,-3.41
1705,2804904,13.83,16.9,-3.07
52,1300029,36.87,39.9,-3.03
1686,2803104,17.19,19.4,-2.21
1682,2802700,24.94,27.0,-2.06
4330,5107107,92.07,90.1,1.97
4185,5004007,80.80,78.9,1.90
72,1302108,75.46,77.3,-1.84
4228,5008404,67.11,65.4,1.71


In [37]:
comparacao = comparacao.merge(
    dim_municipio[["id_municipio", "UF"]],
    on="id_municipio",
    how="left"
)

In [64]:
comparacao.groupby("UF")["diferenca_taxa"].agg(
    ["count", "mean", "min", "max"]
)

,count,mean,min,max
UF,,,,
AC,1,0.220000,0.22,0.22
AL,102,-0.000490,-0.05,0.78
AM,47,-0.190638,-3.03,0.79
AP,16,-0.003125,-0.05,0.04
BA,394,-0.008350,-0.13,0.44
CE,184,-0.004293,-0.05,0.04
ES,78,0.005256,-0.06,0.44
GO,242,-0.025455,-0.79,0.97
MA,216,-0.001435,-0.05,0.12
